# Auction-Based Road Allocation — Interactive Simulation

**Research context.**  
Urban roads are a scarce shared resource.  When too many vehicles compete for the same segment at the same time,
congestion arises.  This notebook implements and compares *online auction mechanisms* for allocating road capacity
in a **time-expanded road network**, where each node is a (physical intersection, time-slot) pair.

**How it works.**  
Vehicles arrive sequentially, each carrying:
- an origin and destination,
- a preferred departure (or arrival) time,
- a maximum willingness to pay (`reserve`), and
- an urgency weight `alpha` that blends price and travel-time in the routing cost.

The mechanism allocates **complete routes** (bundles of segment–time pairs) via
shortest-path search on the time-expanded graph.  If a vehicle's shortest-path
cost exceeds its reserve, the vehicle is rejected.  Accepted vehicles are charged
the path cost and edge prices are updated for subsequent arrivals.

---

## Compared Strategies

| Strategy | Description |
|---|---|
| **Transport-Adapted Pricing** | Exponential price update with per-edge `vmax` scaled by demand/capacity and a travel-time component.  Designed to respect capacity constraints. |
| **Online Competitive** | Exponential update with a global parameter `r` and a global capacity scalar `s_max`.  Follows the BG-style online competitive framework (Buchbinder & Naor, 2009). |
| **Zero Pricing / Free Entry** | Prices are never updated.  Serves as a baseline: all feasible vehicles are accepted at zero toll. |
| **Static Median-Occupancy Pricing** | Runs the dynamic strategy internally for `capacity/2` allocations per edge, then freezes the price.  A static approximation to the dynamic rule. |
| **Smooth Tail** | Exponential on `[0, u0]`, cubic Hermite on `(u0, 1]`.  Reaches `vmax+1` at saturation for dual feasibility, while keeping price growth smooth near the capacity limit. |

## Output Metrics

- **Social welfare** — sum of (reserve − cost) for all served vehicles.
- **Service rate** — fraction of vehicles successfully allocated.
- **Avg travel time** — mean path length (slots) for served vehicles.
- **Avg delay** — mean entry and arrival delay relative to desired time.
- **Revenue** — total tolls collected.
- **Capacity utilisation** — fraction of segment capacity used per time slot.

---
## B · Setup

Run all cells in this section **before** anything else.  
If you are on a fresh Colab runtime, start with **B1** (clone the repo), then run **B2–B4** in order.

In [11]:
# B1 — Clone or update the repository
import os

REPO_ROOT = "/content/transportation-auction-colab"

if not os.path.isdir(REPO_ROOT):
  from google.colab import userdata

  token = userdata.get("cloneCode")
  repo = "transportation-auction-colab"
  user = "CheckIT-App"

  !git clone https://{token}@github.com/{user}/{repo}.git
  #!git clone https://github.com/CheckIT-App/transportation-auction-colab.git {REPO_ROOT}
else:
    print("Repo already present — pulling latest updates ...")
    !git -C {REPO_ROOT} pull

Cloning into 'transportation-auction-colab'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 144 (delta 78), reused 114 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 119.31 KiB | 2.25 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [12]:
# B2 — Install required packages
!pip install -q -r {REPO_ROOT}/requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 98.0 MB/s eta 0:00:00


In [13]:
# B3 — Imports and path setup
import sys, os

REPO_ROOT = "/content/transportation-auction-colab"  # same as B1
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# All scripts must run from the repo root so relative paths (graph files, cache/) resolve correctly.
os.chdir(REPO_ROOT)

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from ExpandedTimeSimulation.simulation_zefat.constants import ALL_STRATEGIES
from ExpandedTimeSimulation.simulation_zefat.colab_utils import (
    load_config_from_widgets,
    preview_network,
    generate_demand_preview,
    plot_demand_distribution,
    run_experiment,
    summarize_results,
    build_summary_tables,
    plot_results,
    export_results,
    decode_reject_reason,
    preview_network_map,
)

print("Setup complete.")

Setup complete.


In [14]:
# B4 — Mount Google Drive (optional but recommended for persistent cache and results)
# If running locally, skip this cell.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
    print("TIP: copy har_nof.gpickle and the cache/ folder into your Drive")
    print("     and update REPO_ROOT / graph_file paths below accordingly.")
except ImportError:
    print("Not running in Colab — Drive mount skipped.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive
TIP: copy har_nof.gpickle and the cache/ folder into your Drive
     and update REPO_ROOT / graph_file paths below accordingly.


---
## C · Configuration

Adjust the sliders and fields below, then **run cell D1** (not this cell) to extract your settings.

> **Important:** Do not re-run *this* cell (C1) after changing sliders — re-running resets all widgets to their default values. Just move on to D1.

> **Tip — minimal first run:** keep `od_count ≤ 200`, `T ≤ 20`, `runs = 1`,
> and select only 2 strategies. Expected runtime on Colab free tier: ~60–90 s.

In [15]:
# C1 — Configuration UI

style = {"description_width": "200px"}
layout = widgets.Layout(width="480px")

# ── Basic parameters ─────────────────────────────────────────────────────
w = {
    # Runs / seed / output
    "num_runs":       widgets.IntSlider(value=1, min=1, max=100, step=1,
                          description="Number of runs", style=style, layout=layout),
    "base_seed":      widgets.IntText(value=2025,
                          description="Random seed", style=style, layout=layout),
    "excel_file":     widgets.Text(value="results.xlsx",
                          description="Output Excel file", style=style, layout=layout),
    # Network
    "graph_file":     widgets.Text(value="har_nof.gpickle",
                          description="Graph file (.gpickle)", style=style, layout=layout),
    "place_name":     widgets.Text(value="Har Nof, Jerusalem, Israel",
                          description="OSM place name", style=style, layout=layout),
    # Time horizon
    "max_time_slots": widgets.IntSlider(value=20, min=10, max=200, step=5,
                          description="Network horizon T (slots)", style=style, layout=layout),
    "peak_slot":      widgets.IntSlider(value=10, min=1, max=100, step=1,
                          description="Peak demand slot", style=style, layout=layout),
    # Pricing
    "vmax":           widgets.FloatSlider(value=100.0, min=10.0, max=500.0, step=10.0,
                          description="vmax (price ceiling)", style=style, layout=layout),
    # Strategies
    "strategy_keys":  widgets.SelectMultiple(
                          options=ALL_STRATEGIES,
                          value=["Zero", "Transport-Adapted Pricing"],
                          description="Strategies", style=style,
                          layout=widgets.Layout(width="480px", height="120px")),
    # Vehicle mode
    "time_mode":      widgets.Dropdown(
                          options=["Both", "Only Entry", "Only Arrival"],
                          value="Both",
                          description="Vehicle mode", style=style, layout=layout),
    "arrival_percentage": widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                          description="Arrival fraction (Both)", style=style, layout=layout),
}

# ── Sweep parameters ─────────────────────────────────────────────────────
# Default: start == end  →  single value, no sweep.
# Set start != end on exactly ONE group to sweep that parameter.
w_sweep = {
    # Vehicles sweep
    "od_count_start": widgets.IntSlider(value=200, min=50, max=5000, step=50,
                          description="Vehicles: start", style=style, layout=layout),
    "od_count_end":   widgets.IntSlider(value=200, min=50, max=5000, step=50,
                          description="Vehicles: end", style=style, layout=layout),
    "od_count_step":  widgets.IntSlider(value=50, min=50, max=1000, step=50,
                          description="Vehicles: step", style=style, layout=layout),
    # Sigma sweep
    "peak_sigma_start": widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5,
                          description="Sigma: start", style=style, layout=layout),
    "peak_sigma_end":   widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5,
                          description="Sigma: end", style=style, layout=layout),
    "peak_sigma_step":  widgets.FloatSlider(value=1.0, min=0.5, max=10.0, step=0.5,
                          description="Sigma: step", style=style, layout=layout),
    # Capacity factor sweep (% of real network capacity)
    "cap_factor_start": widgets.FloatSlider(value=100.0, min=10.0, max=300.0, step=10.0,
                          description="Capacity %: start", style=style, layout=layout),
    "cap_factor_end":   widgets.FloatSlider(value=100.0, min=10.0, max=300.0, step=10.0,
                          description="Capacity %: end", style=style, layout=layout),
    "cap_factor_step":  widgets.FloatSlider(value=10.0, min=5.0, max=100.0, step=5.0,
                          description="Capacity %: step", style=style, layout=layout),
}
w.update(w_sweep)

# ── Advanced parameters (hidden in accordion) ─────────────────────────────
w_adv = {
    "r":                 widgets.IntSlider(value=30, min=5, max=100, step=1,
                             description="r  (Online Competitive base)", style=style, layout=layout),
    "vehicle_T":         widgets.IntSlider(value=100, min=20, max=300, step=10,
                             description="vehicle_T  (max vehicle horizon)", style=style, layout=layout),
    "slot_seconds":      widgets.IntSlider(value=60, min=10, max=300, step=10,
                             description="Seconds per slot", style=style, layout=layout),
    "capacity_is_hourly":widgets.Checkbox(value=True,
                             description="Capacity is hourly", style=style),
    "smooth_tail_u0":    widgets.FloatSlider(value=0.95, min=0.5, max=1.0, step=0.01,
                             description="Smooth Tail u0", style=style, layout=layout),
    "path_solver":       widgets.Dropdown(
                             options=["bidirectional_dijkstra", "dijkstra",
                                      "astar_euclidean", "astar_fflb"],
                             value="bidirectional_dijkstra",
                             description="Path solver", style=style, layout=layout),
    # Alpha range (uniform per-vehicle draw)
    "alpha_lo":          widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05,
                             description="Alpha: min", style=style, layout=layout),
    "alpha_hi":          widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.05,
                             description="Alpha: max", style=style, layout=layout),
    # Entry fee range
    "entry_fee_lo":      widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.5,
                             description="Entry fee: min", style=style, layout=layout),
    "entry_fee_hi":      widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5,
                             description="Entry fee: max", style=style, layout=layout),
    # Lateness fee range
    "lateness_fee_lo":   widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.5,
                             description="Lateness fee: min", style=style, layout=layout),
    "lateness_fee_hi":   widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5,
                             description="Lateness fee: max", style=style, layout=layout),
}
w.update(w_adv)

# ── Sweep validation label ────────────────────────────────────────────────
sweep_warning = widgets.HTML("")

def _check_sweep(*_):
    active = []
    if w["od_count_start"].value != w["od_count_end"].value:
        active.append("Vehicles")
    if w["peak_sigma_start"].value != w["peak_sigma_end"].value:
        active.append("Sigma")
    if w["cap_factor_start"].value != w["cap_factor_end"].value:
        active.append("Capacity %")
    if len(active) > 1:
        sweep_warning.value = (
            f"<span style='color:red'>&#9888; Only one sweep at a time "
            f"&#8212; active: {', '.join(active)}</span>"
        )
    elif len(active) == 1:
        sweep_warning.value = f"<span style='color:green'>&#10003; Sweep: {active[0]}</span>"
    else:
        sweep_warning.value = "<span style='color:grey'>Single run (no sweep)</span>"

_check_sweep()
for _k in ["od_count_start", "od_count_end",
           "peak_sigma_start", "peak_sigma_end",
           "cap_factor_start", "cap_factor_end"]:
    w[_k].observe(_check_sweep, names="value")

# ── Build UI ──────────────────────────────────────────────────────────────
advanced_box = widgets.Accordion(
    children=[widgets.VBox(list(w_adv.values()))],
    selected_index=None,
)
advanced_box.set_title(0, "Advanced settings")

basic_keys = [k for k in w if k not in w_sweep and k not in w_adv]
sweep_box = widgets.VBox(
    [widgets.HTML("<b>── Vehicles sweep</b>")]
    + [w["od_count_start"], w["od_count_end"], w["od_count_step"]]
    + [widgets.HTML("<b>── Peak sigma sweep</b>")]
    + [w["peak_sigma_start"], w["peak_sigma_end"], w["peak_sigma_step"]]
    + [widgets.HTML("<b>── Capacity factor sweep (% of network capacity)</b>")]
    + [w["cap_factor_start"], w["cap_factor_end"], w["cap_factor_step"]]
    + [widgets.HTML("<br>"), sweep_warning]
)
sweep_acc = widgets.Accordion(children=[sweep_box], selected_index=None)
sweep_acc.set_title(0, "Sweep / range settings")

ui = widgets.VBox(
    [widgets.HTML("<h4>Basic parameters</h4>")]
    + [w[k] for k in basic_keys]
    + [widgets.HTML("<br>"), sweep_acc]
    + [widgets.HTML("<br>"), advanced_box]
)
display(ui)


---
## D · Network Preparation

Loads the OSM road network from the `.gpickle` file specified above (or downloads it from OpenStreetMap on first run).
Displays a summary and map of the physical road network.

> **Caching.**  The time-expanded graph is cached automatically to `cache/expanded_net_<hash>.pkl`.
> On Colab, point `graph_file` to a path inside your mounted Drive so the cache persists between sessions.

In [ ]:
# D1 — Extract configuration from widgets
config = load_config_from_widgets(w)
print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k:25s}: {v}")

In [ ]:
# D2 — Preview the road network (no time-expansion yet)
preview_network(config)

In [ ]:
# D3 — Interactive map (folium)
# folium is already installed via requirements.txt (B2), but re-installing is safe
road_map = preview_network_map(config)
display(road_map)


---
## E · Demand Generation

Generates the synthetic vehicle fleet.  Each vehicle gets:
- a random origin–destination pair from the road network,
- a desired entry time drawn from a Gaussian centred on `peak_slot`,
- an `alpha` value (price–time urgency weight) drawn from a three-band mixture,
- a `reserve` price (willingness to pay) computed from alpha and path length.

The preview table shows the first 10 vehicles; the histograms show the full fleet distribution.

In [ ]:
# E1 — Generate vehicle fleet
vehicles, preview_df = generate_demand_preview(config)
print(f"Generated {len(vehicles):,} vehicles.\n")
display(preview_df)

In [ ]:
# E2 — Distribution plots
plot_demand_distribution(vehicles)

---
## F · Simulation

Runs each selected strategy for the configured number of repetitions.
Progress is printed run-by-run.  Results are written to the Excel file
specified in the configuration.

> This cell may take **1–5 minutes** depending on `od_count`, `T`, and the number of strategies.
> Use the minimal configuration (≤ 200 vehicles, T ≤ 20, 2 strategies, 1 run) for a quick test.

In [ ]:
# F1 — Run experiment
excel_path = run_experiment(config)
print(f"\nResults saved to: {excel_path}")

---
## G · Results Summary

Loads all output sheets from the Excel file and builds per-metric comparison tables.
Green highlighting marks the best value per column.

In [ ]:
# G1 — Load result sheets
sheets = summarize_results(excel_path)

# Show a summary table of what was loaded
sheet_info = pd.DataFrame(
    [(name, len(df), len(df.columns)) for name, df in sheets.items()],
    columns=["Sheet", "Rows", "Columns"],
).set_index("Sheet")
display(sheet_info)

In [ ]:
# G2 — Per-metric comparison tables
tables = build_summary_tables(sheets)

for metric, df in tables.items():
    display(
        df.style
          .set_caption(metric)
          .highlight_max(subset=["Mean"], color="#c6efce")
          .format("{:.3f}")
    )


In [ ]:
# G3 — Rejection reason breakdown
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_REJECT_REASON
)

veh_df = sheets.get(SHEET_VEHICLES_TABLE, pd.DataFrame())

if not veh_df.empty and COL_REJECT_REASON in veh_df.columns:
    veh_df = veh_df.copy()
    veh_df["reject_label"] = veh_df[COL_REJECT_REASON].map(decode_reject_reason)
    breakdown = (
        veh_df.groupby([COL_STRATEGY, "reject_label"])
              .size()
              .unstack(fill_value=0)
    )
    breakdown.index.name = "Strategy"
    print("\nVehicle Outcome Breakdown")
    display(breakdown)
else:
    print("vehicles_table sheet is empty or missing reject_reason column.")

---
## H · Visualisations

Plots generated from the saved Excel file — re-run any group independently without re-running the simulation.

Results are organised into six groups, each preceded by a short explanation:

| Group | Contents |
|-------|----------|
| **H1 — Strategy Design** | Theoretical pricing function shapes and price growth |
| **H2 — Social Welfare Scaling** | Social welfare vs fleet size N |
| **H3 — Demand & Network Load** | Requests over time, edge hotspots, vehicles on road, utilisation |
| **H4 — Acceptance Dynamics** | Accepted/rejected per slot, acceptance rate, fee-band breakdown |
| **H5 — Price Dynamics** | Mean & peak edge prices over time per strategy |
| **H6 — Vehicle Economics** | Travel time vs α, delay vs fees, toll vs α, revenue |

> **Note:** Fleet composition histograms → **E2**.  Per-strategy summary tables and rejection counts → **G2–G3**.

### H1 · Strategy Design — Theoretical

*How does each pricing rule behave mathematically?*

These plots are derived from the pricing formulae directly — no simulation data is needed. They give intuition for why strategies differ in their empirical acceptance rates and revenues below.

- **Pricing function shapes** — exponential, smooth-tail, and zero-pricing curves as a function of edge utilisation.
- **Transport-Adapted price growth** — how the recurrence $p_{k+1} = p_k \cdot e^{c/b} + \frac{bt}{T}(e^{c/b}-1)$ grows under different demand and urgency scenarios.

In [ ]:
# H2 — Pricing function shapes and Transport-Adapted price growth (no simulation data needed)
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_pricing_function_shapes,
    plot_transport_adapted_update_growth,
)

selected_strategies = list(config.get("strategy_keys", []))
plot_pricing_function_shapes(strategies=selected_strategies if selected_strategies else None)
plot_transport_adapted_update_growth()

### H3 · Social Welfare Scaling

*How efficiently does each strategy use road capacity as fleet size grows?*

G2 shows aggregate statistics at the simulated N. This chart sweeps across all N values in the results, revealing which mechanisms maintain high social welfare under increasing demand — and how much better they perform compared to the free-entry (Zero) baseline.

In [ ]:
# H4 — Social welfare vs N sweep
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_sw_sweep,
    compute_sw_time_only_from_df,
)

veh_df = sheets["vehicles_table"]
df_sw = compute_sw_time_only_from_df(
    veh_df,
    include_arrival_delay=True,
    include_entry_delay=False,
    served_only=True,
)
plot_sw_sweep(df_sw, baseline="Zero", compare="Transport-Adapted Pricing")

### H5 · Demand & Network Load

*When and where do vehicles compete for road capacity?*

- **Request timeline** — how many vehicles request each time slot under each strategy.
- **Edge heatmap** — which segments are persistent hotspots (top 20 busiest edges × time slots).
- **Vehicles on road** — concurrent active vehicles over the horizon, one line per strategy.
- **Utilisation distribution** — boxplot of edge load; shows whether pricing spreads or concentrates congestion.

In [ ]:
# H6 — Demand and network load
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_requests_over_time,
    plot_requests_heatmap,
    plot_mean_vehicles_on_road,
    plot_utilization_distribution,
)

ts_df = sheets.get("edge_timeslices", pd.DataFrame())

plot_requests_over_time(ts_df)
plot_requests_heatmap(ts_df, top_n=20)
plot_mean_vehicles_on_road(sheets["vehicles_table"])   # all strategies, auto-infers N
plot_utilization_distribution(ts_df)

### H7 · Acceptance & Rejection Dynamics

*Who gets accepted and when?*

G3 shows total rejection counts per strategy. These charts add the **temporal dimension**:
- Stacked counts of accepted / rejected (capacity) / rejected (reserve exceeded) per time slot
- Rolling acceptance rate — does selectivity increase as congestion builds over the horizon?
- Cumulative accepted and rejected — how quickly does each strategy saturate available capacity?
- Acceptance rate by lateness-fee band — are vehicles with tight time constraints systematically crowded out?

In [ ]:
# H8 — Acceptance and rejection dynamics over time
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_accepts_rejects_over_time,
    plot_acceptance_rate_over_time,
    plot_cumulative_accepts_rejects,
    plot_percent_accepted_over_time_by_fee_bands_5000,
)
from ExpandedTimeSimulation.simulation_zefat.constants import COL_LATENESS_FEE

veh_df = sheets["vehicles_table"]

plot_accepts_rejects_over_time(veh_df, by="request")
plot_acceptance_rate_over_time(veh_df, by="request")
plot_cumulative_accepts_rejects(veh_df, by="request")
plot_percent_accepted_over_time_by_fee_bands_5000(
    veh_df,
    bands=((1, 2), (4, 5)),
    panels="bands",
    fee_col=COL_LATENESS_FEE,
    mode="arrival",
)

### H9 · Price Dynamics

*How do edge prices evolve over the simulation horizon?*

Mean and peak prices per time slot, averaged across runs, reveal whether Transport-Adapted pricing smoothly tracks congestion or whether any strategy lets prices spike at peak demand.

In [ ]:
# H10 — Price evolution per strategy
from ExpandedTimeSimulation.simulation_zefat.plots.plots import plot_price_evolution_per_strategy

ts_df = sheets.get("edge_timeslices", pd.DataFrame())

if not ts_df.empty and "N" in ts_df.columns:
    n_vals = pd.to_numeric(ts_df["N"], errors="coerce").dropna()
    common_N = int(n_vals.value_counts().idxmax()) if not n_vals.empty else None
    if common_N is not None:
        plot_price_evolution_per_strategy(ts_df, N_value=common_N, exclude_zero=True)
    else:
        print("Could not infer N — skipping price evolution plot.")
else:
    print("edge_timeslices unavailable or missing N column — skipping.")

### H11 · Vehicle Economics

*What do individual vehicles experience?*

- **Travel time vs α** — do high-urgency vehicles (large α, prefer time over price) end up with shorter routes?
- **Arrival delay vs lateness fee** — does paying a higher penalty actually reduce arrival delay?
- **Toll vs travel time** — do vehicles that pay more in road tolls travel faster?
- **Revenue comparison** — total road toll collected per strategy across fleet sizes (visual companion to G2).
- **Toll vs α** — do urgent vehicles end up on more expensive edges? Consistent across strategies?

> `paid_fee` = **road toll only** (sum of edge prices along the path). Entry/lateness delay fees are routing weights, not monetary payments.

In [ ]:
# H12 — Vehicle economics
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_travel_time_vs_alpha_by_N,
    plot_delay_or_arrival_vs_fee_by_N,
    plot_price_vs_travel_time_by_N,
    plot_revenue_comparison,
    plot_toll_vs_alpha,
)
from ExpandedTimeSimulation.simulation_zefat.constants import COL_PAID_FEE, COL_TRAVEL_TIME

veh_df = sheets["vehicles_table"]
Ns = tuple(sorted(pd.to_numeric(veh_df["N"], errors="coerce").dropna().unique().astype(int)))

plot_travel_time_vs_alpha_by_N(veh_df, Ns=Ns, bins=12)
plot_delay_or_arrival_vs_fee_by_N(veh_df, pair="arrival", Ns=Ns)
plot_price_vs_travel_time_by_N(veh_df, Ns=Ns, price_col=COL_PAID_FEE, travel_time_col=COL_TRAVEL_TIME)
plot_revenue_comparison(veh_df)
plot_toll_vs_alpha(veh_df, Ns=Ns)

### H13 · Custom X/Y Plot

Choose any X and Y metric to compare strategies on a single chart.  
**Sweep-based X axes** (Vehicles, Sigma, Capacity) only work when the last simulation used a sweep — the merged Excel must contain `sweep_param` / `sweep_value` columns.  
**Binned X axes** (Alpha, Entry/Arrival delay cost) are automatically divided into `n_bins` equal-width bins; the line is plotted at each bin midpoint.  
When `num_runs > 1` a ±1 std shaded band is shown.

In [ ]:
# H14 — Custom plot config

X_OPTIONS = {
    "Time slot":           "time_slot",
    "Vehicles (sweep)":    "vehicles_sweep",
    "Sigma (sweep)":       "sigma_sweep",
    "Capacity % (sweep)":  "capacity_sweep",
    "Alpha":               "alpha",
    "Entry delay cost":    "entry_delay_cost",
    "Arrival delay cost":  "arrival_delay_cost",
}
Y_OPTIONS = {
    "Social welfare":      "social_welfare",
    "% Acceptance":        "acceptance",
    "Avg price per route": "avg_price",
    "Avg entry delay":     "avg_entry_delay",
    "Avg arrival delay":   "avg_arrival_delay",
    "Speed":               "speed",
    "Travel time":         "travel_time",
}

w_plot = {
    "x_metric": widgets.Dropdown(
                    options=list(X_OPTIONS),
                    value="Time slot",
                    description="X axis:", style=style, layout=layout),
    "y_metric": widgets.Dropdown(
                    options=list(Y_OPTIONS),
                    value="% Acceptance",
                    description="Y axis:", style=style, layout=layout),
    "n_bins":   widgets.IntSlider(value=10, min=3, max=30, step=1,
                    description="Bins (alpha/cost):", style=style, layout=layout),
    "save_dest": widgets.Dropdown(
                    options=["None", "Download", "Save to Drive", "Both"],
                    value="None",
                    description="Save CSV:", style=style, layout=layout),
    "drive_dir": widgets.Text(
                    value="/content/drive/MyDrive/",
                    description="Drive folder:", style=style, layout=layout),
}
display(widgets.VBox([widgets.HTML("<b>Custom plot settings</b>")] + list(w_plot.values())))


In [ ]:
# H15 — Run custom plot
from ExpandedTimeSimulation.simulation_zefat.colab_utils import plot_custom

_x_key = X_OPTIONS[w_plot["x_metric"].value]
_y_key = Y_OPTIONS[w_plot["y_metric"].value]

_summary_df = plot_custom(
    excel_file=config.get("excel_file", "results.xlsx"),
    x_metric=_x_key,
    y_metric=_y_key,
    strategies=list(config.get("strategy_keys", [])) or None,
    n_bins=w_plot["n_bins"].value,
)

if _summary_df is not None:
    from ExpandedTimeSimulation.simulation_zefat.colab_utils import _X_LABELS, _Y_LABELS
    _x_label = _X_LABELS.get(_x_key, _x_key).lower().replace(" ", "_")
    _y_label = _Y_LABELS.get(_y_key, _y_key).lower().replace(" ", "_")
    _csv_name = f"custom_plot_{_x_label}_vs_{_y_label}.csv"
    _dest = w_plot["save_dest"].value

    if _dest != "None":
        try:
            import tempfile, os
            from google.colab import files

            if _dest in ("Save to Drive", "Both"):
                _drive_path = os.path.join(w_plot["drive_dir"].value.rstrip("/"), _csv_name)
                _summary_df.to_csv(_drive_path, index=False)
                print(f"Saved to Drive: {_drive_path}")

            if _dest in ("Download", "Both"):
                _tmp = os.path.join(tempfile.gettempdir(), _csv_name)
                _summary_df.to_csv(_tmp, index=False)
                files.download(_tmp)

        except ImportError:
            print(f"Not running in Colab — displaying DataFrame (save manually as {_csv_name}):")
            display(_summary_df)


---
## I · Export

Saves each result sheet as a CSV file and writes the experiment configuration as JSON.
The Excel file is already written by the simulation step.

In [ ]:
# I1 — Export CSVs + config JSON
export_results(excel_path, config, output_dir="experiment_output")

---
## J · Developer / Advanced Tools

Low-level inspection cells.  Useful for debugging, understanding individual vehicle
allocations, tracing edge prices, or benchmarking path solvers.  These cells are
independent — run them in any order after the simulation has completed.

In [ ]:
# J1 — Inspect a single vehicle allocation
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_SERVED,
    COL_ALPHA, COL_RESERVE, COL_PAID_FEE,
    COL_TRAVEL_TIME, COL_ENTRY_DELAY, COL_ARRIVAL_DELAY,
    COL_REJECT_REASON,
)

VEHICLE_IDX = 0        # <-- change this to inspect a different vehicle
STRATEGY    = None     # <-- set to a strategy name string, or None for the first available

veh_df = sheets.get(SHEET_VEHICLES_TABLE, pd.DataFrame())
if veh_df.empty:
    print("vehicles_table is empty — run F1 first.")
else:
    strats = veh_df[COL_STRATEGY].unique()
    chosen_strat = STRATEGY if STRATEGY in strats else strats[0]
    subset = veh_df[veh_df[COL_STRATEGY] == chosen_strat]
    row = subset.iloc[VEHICLE_IDX]
    print(f"Vehicle #{VEHICLE_IDX}  —  strategy: {chosen_strat}")
    print(f"  served          : {row.get(COL_SERVED)}")
    print(f"  reject reason   : {decode_reject_reason(row.get(COL_REJECT_REASON, 0))}")
    print(f"  alpha           : {row.get(COL_ALPHA, '—'):.3f}")
    print(f"  reserve         : {row.get(COL_RESERVE, '—'):.3f}")
    print(f"  paid fee        : {row.get(COL_PAID_FEE, '—')}")
    print(f"  travel time     : {row.get(COL_TRAVEL_TIME, '—')} slots")
    print(f"  entry delay     : {row.get(COL_ENTRY_DELAY, '—')} slots")
    print(f"  arrival delay   : {row.get(COL_ARRIVAL_DELAY, '—')} slots")

In [ ]:
# J2 — Inspect price evolution for a single edge over time
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_EDGE_TIMESLICES, COL_STRATEGY, COL_T, COL_PRICE, COL_UTIL, COL_EDGE
)

EDGE_IDX = 0    # <-- index into the list of unique edges

ts_df = sheets.get(SHEET_EDGE_TIMESLICES, pd.DataFrame())
if ts_df.empty:
    print("edge_timeslices is empty — run F1 first.")
else:
    unique_edges = ts_df[COL_EDGE].unique()
    if EDGE_IDX >= len(unique_edges):
        print(f"EDGE_IDX={EDGE_IDX} out of range. There are {len(unique_edges)} unique edges.")
    else:
        chosen_edge = unique_edges[EDGE_IDX]
        subset = ts_df[ts_df[COL_EDGE] == chosen_edge]

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for strat, grp in subset.groupby(COL_STRATEGY):
            grp_sorted = grp.sort_values(COL_T)
            axes[0].plot(grp_sorted[COL_T], grp_sorted[COL_PRICE], label=strat, marker="o", ms=3)
            axes[1].plot(grp_sorted[COL_T], grp_sorted[COL_UTIL],  label=strat, marker="o", ms=3)

        axes[0].set_title(f"Price over time — edge {EDGE_IDX}", fontsize=12)
        axes[0].set_xlabel("Time slot"); axes[0].set_ylabel("Price")
        axes[0].legend(fontsize=9)

        axes[1].set_title(f"Utilisation over time — edge {EDGE_IDX}", fontsize=12)
        axes[1].set_xlabel("Time slot"); axes[1].set_ylabel("Utilisation")
        axes[1].axhline(1.0, color="red", linestyle="--", linewidth=0.8, label="capacity")
        axes[1].legend(fontsize=9)

        plt.tight_layout()
        plt.show()
        print(f"Edge: {chosen_edge}")

In [ ]:
# J3 — Profile runtime for a minimal run
import time

PROFILE_CONFIG = {
    "od_count": 100,
    "num_runs": 1,
    "max_time_slots": 15,
    "vmax": 100.0,
    "r": 30,
    "slot_seconds": 60,
    "vehicle_T": 50,
    "peak_slot": 8,
    "peak_sigma": 3.0,
    "strategy_keys": ["Zero", "Transport-Adapted Pricing"],
    "base_seed": 42,
    "excel_file": "_profile_run.xlsx",
    "graph_file": config.get("graph_file", "har_nof.gpickle"),
    "place_name": config.get("place_name", "Har Nof, Jerusalem, Israel"),
    "capacity_is_hourly": True,
    "smooth_tail_u0": 0.95,
}

t0 = time.perf_counter()
run_experiment(PROFILE_CONFIG)
elapsed = time.perf_counter() - t0
print(f"\nProfiling complete: {elapsed:.1f}s  ({PROFILE_CONFIG['od_count']} vehicles, "
      f"T={PROFILE_CONFIG['max_time_slots']}, {len(PROFILE_CONFIG['strategy_keys'])} strategies)")

In [ ]:
# J4 — Path-solver speed benchmark
# Runs the same 100-vehicle problem with Zero pricing under four routing algorithms
# and reports total wall time, throughput, and a bar chart.
import time, copy
import matplotlib.pyplot as plt
import pandas as pd

from ExpandedTimeSimulation.simulation_zefat.experiments.batch_run import (
    _load_or_build_graph_and_xy,
)
from ExpandedTimeSimulation.simulation_zefat.network import TimeExpandedRoadNetwork
from ExpandedTimeSimulation.simulation_zefat.auction_simulator import AuctionSimulator
from ExpandedTimeSimulation.simulation_zefat.strategy_factory import make_strategy
from ExpandedTimeSimulation.simulation_zefat.experiments.vehicle_generation import (
    mixed_alpha_sampler, assign_peak_desired_entry, PeakSchedule,
)

# ── Parameters (edit here) ────────────────────────────────────────────────
SOLVERS      = ["dijkstra", "bidirectional_dijkstra", "astar_euclidean", "reverse_astar_ff"]
N_VEHICLES   = 100
T            = 15
VMAX         = 100.0
GRAPH_FILE   = config.get("graph_file",  "har_nof.gpickle")
PLACE_NAME   = config.get("place_name", "Har Nof, Jerusalem, Israel")
SLOT_SECONDS = int(config.get("slot_seconds", 60))

# ── Build graph + vehicles once ───────────────────────────────────────────
import os
node_xy_file = os.path.splitext(GRAPH_FILE)[0] + "_node_xy_meters.pkl"

loader, node_xy_file = _load_or_build_graph_and_xy(
    place_name=PLACE_NAME,
    graph_file=GRAPH_FILE,
    node_xy_file=node_xy_file,
    time_slot_duration=SLOT_SECONDS,
    od_count=N_VEHICLES,
)
loader.generate_od_pairs()
base_edges = loader.convert_to_base_edges()

alpha_mix = mixed_alpha_sampler([(0.4,(0.0,0.3)),(0.4,(0.3,0.7)),(0.2,(0.7,1.0))])
vehicles  = loader.generate_vehicles(alpha=alpha_mix)
assign_peak_desired_entry(
    vehicles,
    schedule=PeakSchedule(peak_slot=8, sigma=3.0, horizon_T=50),
    write_arrival=True,
)
print(f"Fleet: {len(vehicles)} vehicles, T={T}, graph loaded.")

# ── Benchmark loop ────────────────────────────────────────────────────────
results = []
for solver in SOLVERS:
    net = TimeExpandedRoadNetwork(
        base_edges,
        max_time_slots=T,
        vmax=VMAX,
        r=30,
        pricing_strategy=make_strategy("Zero"),
        capacity_is_hourly=bool(config.get("capacity_is_hourly", True)),
        slot_seconds=SLOT_SECONDS,
        node_xy_file=node_xy_file,
    )
    sim = AuctionSimulator(net, copy.deepcopy(vehicles), path_solver=solver)
    t0 = time.perf_counter()
    sim.run()
    elapsed = time.perf_counter() - t0
    served  = sum(1 for v in sim.vehicles if v.get("served"))
    ms_per_v = elapsed / max(len(vehicles), 1) * 1000
    results.append({"Solver": solver, "Time (s)": round(elapsed, 2),
                    "ms / vehicle": round(ms_per_v, 1), "Served": served})
    print(f"  {solver:30s}  {elapsed:6.2f}s  ({ms_per_v:.1f} ms/veh)  served={served}/{len(vehicles)}")

# ── Table ─────────────────────────────────────────────────────────────────
df_res = pd.DataFrame(results).set_index("Solver")
display(df_res.style.highlight_min(subset=["Time (s)", "ms / vehicle"], color="#c6efce")
              .highlight_max(subset=["Served"], color="#c6efce"))

# ── Bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
bars = ax.bar(df_res.index, df_res["Time (s)"],
              color=colors[:len(df_res)], edgecolor="none", alpha=0.85)
ax.bar_label(bars, fmt="%.2fs", padding=3, fontsize=10)
ax.set_title(f"Path-Solver Runtime  ({N_VEHICLES} vehicles, T={T})", fontsize=13)
ax.set_ylabel("Wall time (seconds)", fontsize=11)
ax.set_xlabel("Algorithm", fontsize=11)
ax.tick_params(axis="x", labelsize=10)
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
